# M3L4 E02 — Tracing de un sistema multiagente mock
### Modulo 3 · Lecture 4 · Construccion, pruebas y trazabilidad de agentes en produccion

---

## Que necesitas saber antes

| Modulo | Concepto | Por que lo necesitas aca |
|---|---|---|
| M3L4 E01 | MiniTracer: start_trace, add_span, update_trace_output | Lo vas a usar exactamente igual aca |
| M3L4 E00 | Trace, Span, jerarquia | Aplicas los conceptos a un sistema multi-agente real |
| M3L3 | Sistemas multi-agente con orquestador | La arquitectura de routing es la misma |
| M3L1 | Tool contracts | Cada agente devuelve un contrato estable (texto) |

Si no completaste E01, hace eso primero. Aca usamos MiniTracer ya funcional.

---

## Definiciones clave

| Concepto | Definicion simple | Como aparece en este notebook |
|---|---|---|
| **Orquestador** | Componente que decide a que agente enviar cada request | `route_query()` clasifica el intent y devuelve `'hr'`, `'it'`, etc. |
| **Agente especialista** | Funcion que resuelve un dominio especifico (HR, IT, Finance, Legal) | `hr_agent()`, `it_agent()`, `finance_agent()`, `legal_agent()` |
| **Trazar (instrumentar)** | Agregar codigo de tracing a un sistema existente | Llamar `tracer.start_trace()`, `tracer.add_span()` dentro de `handle_query_with_tracing()` |
| **Span de routing** | Span que registra la clasificacion del intent | `tracer.add_span(trace, 'orchestrator-routing', ...)` |
| **Span de agente** | Span que registra la ejecucion del agente especialista | `tracer.add_span(trace, '{intent}-agent', ...)` |
| **Tiempo real** | Duracion medida con `time.time()` al ejecutar | `start = time.time()` antes de llamar, `duration_ms` al terminar |

---

## Como encaja esto en un sistema de agentes

```
Query del usuario
    |
    v
route_query()  --------->  orquestador-routing (span)
    |                          | intent detectado
    v                          | duracion real
agentes especialistas
    |  hr_agent()              |
    |  it_agent()       ----->  {intent}-agent (span)
    |  finance_agent()          | respuesta generada
    |  legal_agent()            | duracion real
    v
Respuesta final  --------->  trace['output']
```

**Objetivo del ejercicio:** aplicar el MiniTracer a un sistema multiagente real con orquestador y agentes especialistas.

## Instalacion e imports

Al igual que E01, este notebook solo usa la biblioteca estandar de Python:

| Import | Que hace | Por que lo necesitamos |
|---|---|---|
| `import time` | `time.time()` para medir duracion real | Medir cuantos ms tarda el routing y cada agente |
| `import uuid` | `uuid.uuid4()` para identificadores unicos | Generar trace_id y span_id |
| `from datetime import datetime` | `datetime.utcnow().isoformat()` | Timestamp de creacion del trace |

```python
import time
import uuid
from datetime import datetime
```

## Paso 1 — Copiar el MiniTracer del E01

> Copiamos la clase completa para que el notebook sea autocontenido.

Si completaste los TODO de E01, esta clase ya deberia funcionar. Si no, usa la version completa de abajo.

In [ ]:
import time
import uuid
from datetime import datetime

class MiniTracer:
    def __init__(self):
        self.traces = []

    def start_trace(self, name, input_data=None, metadata=None, tags=None):
        trace = {
            'trace_id': str(uuid.uuid4()),
            'name': name,
            'input': input_data,
            'output': None,
            'metadata': metadata or {},
            'tags': tags or [],
            'spans': [],
            'created_at': datetime.utcnow().isoformat(),
            '_started_at_ts': time.time()
        }
        self.traces.append(trace)
        return trace

    def add_span(self, trace, name, input_data=None, output_data=None, metadata=None, duration_ms=None):
        span = {
            'span_id': str(uuid.uuid4()),
            'name': name,
            'input': input_data,
            'output': output_data,
            'metadata': metadata or {},
            'duration_ms': duration_ms
        }
        trace['spans'].append(span)
        return span

    def update_trace_output(self, trace, output_data):
        trace['output'] = output_data
        trace['total_duration_ms'] = round((time.time() - trace['_started_at_ts']) * 1000, 2)

    def show_trace(self, trace):
        return trace

print('MiniTracer listo.')

## Paso 2 — Sistema multiagente (ya dado)

El router y los agentes ya estan implementados. Tu tarea es **instrumentarlos con trazas**.

### Arquitectura del sistema

```
route_query(query)
    |
    +-- contiene palabras de RRHH?  -> 'hr'
    +-- contiene palabras de IT?     -> 'it'
    +-- contiene palabras de Finance? -> 'finance'
    +-- contiene palabras de Legal?  -> 'legal'
    +-- menos de 3 palabras?         -> 'clarification'
    +-- si no                          -> 'general'
```

Cada agente especialista recibe el query y devuelve un texto de respuesta. El diccionario `agents` mapea cada intent a su funcion.

In [ ]:
def route_query(query: str) -> str:
    q = query.lower()
    if any(w in q for w in ['vacaciones', 'licencia', 'recibo', 'nomina', 'portal rrhh']):
        return 'hr'
    if any(w in q for w in ['vpn', 'laptop', 'error', 'app', 'wifi', 'login', 'contrasena']):
        return 'it'
    if any(w in q for w in ['factura', 'pago', 'reembolso', 'gasto', 'cobro']):
        return 'finance'
    if any(w in q for w in ['contrato', 'legal', 'confidencialidad', 'nda']):
        return 'legal'
    if len(q.split()) <= 2:
        return 'clarification'
    return 'general'

def hr_agent(query: str) -> str:
    return 'Soy HRAgent. Te ayudo con vacaciones, licencias, recibos y temas de RRHH.'

def it_agent(query: str) -> str:
    return 'Soy ITAgent. Te ayudo con VPN, laptop, accesos y errores tecnicos.'

def finance_agent(query: str) -> str:
    return 'Soy FinanceAgent. Te ayudo con facturas, pagos y reembolsos.'

def legal_agent(query: str) -> str:
    return 'Soy LegalAgent. Te ayudo con contratos y temas legales.'

def general_agent(query: str) -> str:
    return 'Necesito mas contexto para ayudarte correctamente.'

agents = {
    'hr': hr_agent,
    'it': it_agent,
    'finance': finance_agent,
    'legal': legal_agent,
    'general': general_agent,
    'clarification': general_agent
}

print('Agentes listos.')

## Paso 3 — TODO: Instrumentar con trazas

Completa la funcion `handle_query_with_tracing` para que:

1. **Cree un trace** al inicio con el query
2. **Agregue un span** para el routing (con intent detectado y duracion real)
3. **Agregue un span** para el agente seleccionado (con respuesta y duracion real)
4. **Actualice el output** del trace al final

### Desglose de `handle_query_with_tracing(query, tracer)`

| Parametro | Tipo | Que es |
|---|---|---|
| `query` | `str` | La consulta del usuario (ej: 'No puedo ver mi factura') |
| `tracer` | `MiniTracer` | Instancia del tracer que crea y almacena traces |
| **Retorna** | `dict` | El trace completo con todos los spans |

### Dentro de la funcion: flujo esperado

```python
def handle_query_with_tracing(query, tracer):
    # 1. Crear trace
    trace = tracer.start_trace(name, input_data, metadata, tags)

    # 2. Medir routing
    start = time.time()
    intent = route_query(query)
    duration = round((time.time() - start) * 1000, 2)
    tracer.add_span(trace, 'orchestrator-routing', input, output, metadata, duration)

    # 3. Medir agente
    start = time.time()
    response = agents[intent](query)
    duration = round((time.time() - start) * 1000, 2)
    tracer.add_span(trace, f'{intent}-agent', input, output, metadata, duration)

    # 4. Actualizar output
    tracer.update_trace_output(trace, output)

    return trace
```

In [ ]:
def handle_query_with_tracing(query: str, tracer: MiniTracer) -> dict:
    """
    Procesa una query con un sistema multiagente e instrumenta cada paso con trazas.

    Args:
        query: consulta del usuario
        tracer: instancia de MiniTracer

    Returns:
        trace: el trace completo con todos los spans
    """
    # TODO 1: crear el trace con nombre 'multiagent-support-request'
    # metadata: {'environment': 'notebook', 'router_version': 'v1'}
    # tags: ['multiagent', 'm3l4']
    trace = None  # reemplazar

    # TODO 2: medir el tiempo real del routing y agregar span 'orchestrator-routing'
    # input: {'query': query}
    # output: {'intent': intent}
    # metadata: {'router_version': 'v1'}
    intent = None  # reemplazar con route_query(query)

    # TODO 3: medir el tiempo real del agente y agregar span '{intent}-agent'
    # input: {'query': query}
    # output: {'response': response}
    response = None  # reemplazar con agents.get(intent, general_agent)(query)

    # TODO 4: actualizar el output del trace
    # {'intent': intent, 'final_response': response}

    return trace

print('Funcion definida.')

In [ ]:
tracer = MiniTracer()

trace = handle_query_with_tracing('No puedo ver mi factura', tracer)
trace

## Paso 4 — Ejecutar multiples consultas

Probamos el sistema con diferentes tipos de consultas para ver como el tracer registra cada una.

In [ ]:
queries = [
    'Como solicito mis dias de vacaciones?',
    'Mi VPN no conecta desde ayer',
    'Necesito ver mi factura del mes pasado',
    'Necesito el contrato de confidencialidad actualizado',
    'ayuda'
]

tracer2 = MiniTracer()
for q in queries:
    t = handle_query_with_tracing(q, tracer2)
    if t:
        print(f"Query: {q[:45]}...")
        print(f"  Intent: {t['output']['intent'] if t['output'] else 'N/A'}")
        print(f"  Spans: {[s['name'] for s in t['spans']]}")
        print()

## Checks automaticos

Estos asserts validan que `handle_query_with_tracing` funciona correctamente.

In [ ]:
tracer_check = MiniTracer()

t1 = handle_query_with_tracing('No puedo ver mi factura', tracer_check)
assert t1 is not None, 'trace es None'
assert len(t1['spans']) == 2, f'Se esperaban 2 spans, hay {len(t1["spans"])}'
assert t1['spans'][0]['name'] == 'orchestrator-routing', 'Primer span incorrecto'
assert t1['output']['intent'] == 'finance', f"Intent incorrecto: {t1['output']['intent']}"
assert t1['spans'][0]['output']['intent'] == 'finance'
assert t1['spans'][1]['output']['response'] is not None
assert t1['spans'][0]['duration_ms'] is not None
assert t1['spans'][1]['duration_ms'] is not None

print('Checks E02 OK')

## Errores comunes

| Error | Causa | Como detectarlo |
|---|---|---|
| `intent` siempre es `None` | No llamar `route_query(query)` | El span de routing no tiene output util |
| `response` siempre es `None` | No llamar `agents.get(intent)(query)` | El span del agente no tiene respuesta |
| Un solo span en vez de dos | No agregar el span de routing o el de agente | `len(trace['spans'])` es 1 en vez de 2 |
| `duration_ms` es `None` | No medir tiempo real con `time.time()` | Los spans existen pero no tienen metrica |
| `trace['output']` es `None` | No llamar `update_trace_output()` | El trace queda incompleto |
| El trace no tiene `total_duration_ms` | `update_trace_output()` no calcula duracion total | Falta la metrica global del sistema |

## Sintesis

### Que construiste

| Componente | Descripcion |
|---|---|
| `handle_query_with_tracing()` | Funcion que envuelve el sistema multi-agente y registra cada paso |
| Span de routing | Registra que intent se detecto y cuanto tardo la clasificacion |
| Span de agente | Registra que agente respondio, la respuesta y su duracion |
| Trace completo | Unifica toda la informacion de una request en una sola estructura |

### Diferencia con E01

| Aspecto | E01 (MiniTracer basico) | E02 (Multi-agente con tracing) |
|---|---|---|
| Spans | Fijos, definidos manualmente | Dinamicos, dependen del intent |
| Duracion | Pasada como argumento fijo | Medida con `time.time()` real |
| Sistema trazado | MiniTracer solo | Sistema multi-agente completo |
| Output | Respuesta fija | Depende del agente especialista |

### Relacion con otros ejercicios

| Ejercicio | Conexion con E02 |
|---|---|
| **E03** | Tomar estos traces y diagnosticar fallas automaticamente |
| **E04** | Golden datasets para evaluar routing vs ground truth |
| **E08** | LangGraph + Langfuse: tracing automatico sin instrumentacion manual |